#### Simple Chatbot Without LangChain using Groq api

In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq

# Load environment variables from .env file
load_dotenv()

client = Groq()
completion = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
      {
        "role": "user",
        "content": "Hello! I am alok"
      }
    ],
    temperature=1,
    max_completion_tokens=2048,
    top_p=1,
    reasoning_effort="medium",
    stream=True,
    stop=None
)

for chunk in completion:
    print(chunk.choices[0].delta.content or "", end="")




#### Simple Q&A bot with LangChain wiht Groq

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b")

response = llm.invoke("Hello, How are you ?")

print(response.content)

#### Common LLM calls with - `init_chat_model`

In [ ]:
from langchain.chat_models import init_chat_model

model = "groq:openai/gpt-oss-120b"
# "openai:gpt-6-astra" - Using init_chat_model we can switch to any model instantly

llm = init_chat_model(model) 

response = llm.invoke("Hello")
print(response.content)

In [ ]:
from langchain.chat_models import init_chat_model

model = "openai/gpt-oss-120b"

llm = init_chat_model(model, model_provider="groq") 

response = llm.invoke("Hello, give the angry emoji")
print(response.content)

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

MODEL = os.getenv("MODEL", "openai/gpt-oss-120b")
PROVIDER = os.getenv("PROVIDER", "groq")

llm = init_chat_model(MODEL, model_provider=PROVIDER) 

response = llm.invoke("Hello, give the angry emoji")
print(response.content)


#### Code components of `LangChain`

##### 1. Messages

#### Prompt -> LLM -> 3 Types -> Role of prompt ["User", "ai", "system"]

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

MODEL = os.getenv("MODEL", "openai/gpt-oss-120b")
PROVIDER = os.getenv("PROVIDER", "groq")

llm = init_chat_model(MODEL, model_provider=PROVIDER) 

res = llm.invoke([
    {"role": "system", "content": "You are a python developr having 7 years of experience in python ,but don't have any idea about java."} ,
    {"role": "user", "content": "How to find the 2nd laregest number in array in java"} ,
])

print(res.content)


In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

prompts = [
    SystemMessage(content="You are a python developr having 7 years of experience in python ,but don't have any idea about java not even a single thing and you can't answer anything related to java programming."),
    HumanMessage(content="How to find the 2nd laregest number in array in java")
]

res = llm.invoke(prompts)

print(res.content)

##### 2. Prompt Templates

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    {"role":"system", "content":"You are a 7 years of {language} developer"}, 
    {"role":"user", "content":"{query}"}, 
])

In [ ]:
prompt

In [ ]:
prompt = prompt.invoke({"language": "javascript", "query": "How to find the 2nd laregest number in array"})
prompt

#### 3. LCLE - `|` Operator

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 1. Define the prompt template with your placeholders
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful programming assistant specialized in {language}."),
    ("human", "{query}")
])

# 2. Supply the variables
question = "How to map the array"
language = "javascript"

# 3. Format and pass to the model
finalprompt = prompt_template.invoke({"language": language, "query": question})

res = llm.invoke(finalprompt)
print(res.content)


#### `LangChain` Chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Ensure you have the ChatPromptTemplate object
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Reply in {language}."),
    ("human", "{query}")
])

qna_chain = prompt | llm 

res = qna_chain.invoke({"language": "english", "query": "what is gen-ai"})
print(res.content)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Ensure you have the ChatPromptTemplate object
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Reply in {language}."),
    ("human", "{query}")
])

# 2. Custom helper functions
def getContent(res):
    return res.content


qna_chain = prompt_template | llm | getContent

# 4. Invoke the chain
res = qna_chain.invoke({"language": "english", "query": "what is gen-ai"})
print(res)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.messages import AIMessage

# 1. Ensure you have the ChatPromptTemplate object
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Reply in {language}."),
    ("human", "{query}")
])

# 2. Custom helper functions
def getContent(res: AIMessage):
    return res.content


qna_chain = prompt_template | llm | getContent

# 4. Invoke the chain
res = qna_chain.invoke({"language": "english", "query": "what is gen-ai"})
print(res)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Ensure you have the ChatPromptTemplate object
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Reply in {language}."),
    ("human", "{query}")
])

# 2. Custom helper functions
def getContent(res):
    return res.content

def shortTheMessage(msg: str):
    return msg[:100]

# 3. Build the chain with prompt_template (NOT prompt)
qna_chain = prompt_template | llm | StrOutputParser() | shortTheMessage

# 4. Invoke the chain
res = qna_chain.invoke({"language": "english", "query": "what is gen-ai"})
print(res)


#### 4. `Memory` - Conversation History

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-20b")

In [ ]:
res = llm.invoke("Who is Pm of india ?")
print(res.content)

In [ ]:
res = llm.invoke("What is his age ?")
print(res.content)

#### Implement history...

In [ ]:
history = []

In [ ]:
while True:
    query = input("User: ")
    if query == "exit":
        break
    
    history.append(HumanMessage(query))
    res = llm.invoke(history)
    history.append(AIMessage(content=res.content))
    print("AI: ", res.content)

In [ ]:
history

#### 5. `Structured output`

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-120b")

In [ ]:
res = llm.invoke("Hello, My name is Alok Hotta, age is 10 and email is alokhotta10@gmail.com, Can you please parse the name, email, and age in the form of json.")

In [ ]:
print(res.content)

In [ ]:
### Solution using pydantic classes

from pydantic import BaseModel, Field

class User(BaseModel):
    name:str = Field(description="User full name")
    age:int = Field(description="User age in years")
    email:str = Field(description="User email address")

In [ ]:
llm_with_struct_out = llm.with_structured_output(User)

In [ ]:
res = llm_with_struct_out.invoke(
    "Extract the details for: My name is Alok Hotta, age is 25, and email is alokhotta10@gmail.com."
)

In [ ]:
res

In [ ]:
print(res)

In [ ]:
print(res.name)
print(res.age)
print(res.email)

In [ ]:
res.model_dump()

#### 6. `Streaming`

In [ ]:
from langchain_groq import ChatGroq
from time import sleep
llm = ChatGroq(model="openai/gpt-oss-20b", streaming=True)

In [ ]:
res = llm.stream("Explain me Machine learning in 200 words")

In [ ]:
print(res)

In [ ]:
for chunk in res:
    print(chunk.content, end='', flush=True)


#### 7. `AI Agents`

7.1 Defining `tools`

In [ ]:
from langchain.tools import tool

@tool
def send_email(email:str):
    """ I can send email to any email address  """
    print("[TOOL] Called", email)
    return "Email Sent Successfully"

@tool
def add_numbers(a:int, b:int):
    """Add two number and return the sum"""
    return a + b

@tool
def multiply_numbers(a:int, b:int):
    """Multiply two number and return the product"""
    return a * b

In [ ]:
send_email.invoke({"email":"email@gmail.com"})

In [ ]:
add_numbers.invoke({"a": 10, "b": 20})

In [ ]:
multiply_numbers.invoke({"a": 10, "b": 20})

7.2 Agent - `create_agent`

In [ ]:
from langchain.agents import create_agent

In [ ]:
agent = create_agent(
    system_prompt="""
        You always call tools for user question. 
    """,
    model=llm,
    tools=[send_email, add_numbers, multiply_numbers]
)

In [ ]:
agent

In [ ]:
question = "I want to send a leave request to this mail - alokhotta10@gmail.com"
res = agent.invoke({"messages": [{"role": "user", "content": question}] })

In [ ]:
for msg in res["messages"]:
    msg.pretty_print()

In [ ]:
que = "What is (2 + 3) * 7"
res = agent.invoke({ "messages": [ {"role":"user", "content":que} ] })

In [ ]:
for msg in res["messages"]:
    msg.pretty_print()

##### 7.3 Agent - `memory`

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
agent = create_agent(
    system_prompt="""
        You always call tools for user question. 
    """,
    model=llm,
    tools=[send_email, add_numbers, multiply_numbers],
    checkpointer=InMemorySaver()
)

In [ ]:
while True:
    query = input("User: ")
    if query == "exit":
        break
    
    res = agent.invoke(
        { "messages": [ {"role":"user", "content":query} ] },
        {"configurable": {"thread_id": "user_1"}}
        )
    
    ans = res["messages"][-1].content
    print("AI: ", ans)

#### 8. Prebuilt middleware


##### `Summarization` middleware

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware

CHAT = [
    "My order id is ORD-4471.",
    "It was a Nikon Z50 camera.",
    "The box came open.",
    "The lens cap is missing.",
    "I want a replacement, not a refund.",
    'I love react js, and want to learn it',
    'Who is PM of India ?'
]

agent = create_agent(
    system_prompt="""
        You are a helpfull assistant, reply in one short line
    """,
    model=llm,
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("messages", 6),
            keep=("messages", 2)
        )
    ]
)

In [ ]:
config = {"configurable": {"thread_id": "user_2"}}

for msg in CHAT:
    agent.invoke(
        { "messages": [ {"role":"user", "content":msg} ] },
        config=config
    )
    
    print("Messages in history: ", len(agent.get_state(config).values["messages"]))

In [ ]:
agent.get_state(config).values["messages"]

In [ ]:
res = agent.invoke(
    { "messages": [ {"role":"user", "content":"What is my order id"} ] },
    config=config
)

In [ ]:
print(res)

In [ ]:
print(res["messages"][-1].content)

In [ ]:
res = agent.invoke(
    { "messages": [ {"role":"user", "content":"What is my camera name"} ] },
    config=config
)

In [ ]:
print(res)

In [ ]:
print(res["messages"][-1].content)

#### PIIMiddleware

In [94]:
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model=llm,
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
    ],
)

In [95]:
query = "Hello, I received email from prateek@gmail.com for the payment of credit card 4111 1111 1111 1111"
res = agent.invoke({
    "messages": [ {"role":"user", "content":query} ]
})

In [96]:
res["messages"][0].content

'Hello, I received email from [REDACTED_EMAIL] for the payment of credit card **** **** **** 1111'